
# Ionizing-photon production rate vs SSP age

The hydrogen-ionizing photon production rate ``Q_H`` of a simple
stellar population drops by ~5 dex from 1 Myr to 100 Myr as O stars
die. Different SSP libraries predict different ``Q_H(t)`` because
they differ in upper-IMF treatment, stellar rotation, and (most
dramatically) whether massive binaries are included — BPASS extends
the ``Q_H``-producing phase to ~30 Myr.

We integrate ``L_ν / (h ν)`` blueward of the Lyman limit (912 Å) at
each SSP age at solar metallicity for four bundled bare-stellar
grids and overlay the curves.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

H_PLANCK = 6.626e-27  # erg s
C_AA_PER_S = 2.998e18  # Å / s
L_SUN_ERG_S = 3.839e33  # erg / s  (IAU 2015)

GRIDS = [
    ("fsps_prsc_miles_chabrier", "FSPS-Padova / MILES (Chabrier)"),
    ("fsps_mist_c3k_a_chabrier", "FSPS-MIST / C3K (Chabrier)"),
    ("bpss_stars_c3k_a_chabrier", "BPASS stars-only / C3K"),
    ("bc03_pdva_stelib_chabrier", "BC03-Padova / STELIB"),
]
COLORS = plt.cm.viridis(np.linspace(0.05, 0.92, len(GRIDS)))


def _q_h_per_age(ssp) -> tuple[np.ndarray, np.ndarray]:
    """Return ``(ages_yr, Q_H_per_Msun)`` at solar metallicity.

    The bundled SSP cubes store ``L_ν`` in ``L_sun / Hz / Msun``; we
    convert to ``erg / s / Hz / Msun`` and integrate the photon rate
    blueward of 912 Å.
    """
    wave = np.asarray(ssp.ssp_wave, dtype=np.float64)
    log_age_gyr = np.asarray(ssp.ssp_lg_age_gyr, dtype=np.float64)
    log_zmet = np.asarray(ssp.ssp_lgmet, dtype=np.float64)
    zsol_idx = int(np.argmin(np.abs(log_zmet - (-1.85))))
    flux = np.asarray(ssp.ssp_flux, dtype=np.float64)
    l_nu = flux[zsol_idx, :, :] if flux.ndim == 3 else flux[zsol_idx, :, :, 0]
    l_nu = l_nu * L_SUN_ERG_S  # → erg/s/Hz/Msun

    # Some grids store age=0 as log_age=-inf — drop those rows.
    keep = np.isfinite(log_age_gyr)
    log_age_gyr = log_age_gyr[keep]
    l_nu = l_nu[keep]

    ages_yr = 10.0 ** (log_age_gyr + 9.0)
    mask = (wave <= 912.0) & (wave > 0)
    nu = C_AA_PER_S / wave[mask]
    order = np.argsort(nu)
    nu_s = nu[order]
    q_h = np.empty(l_nu.shape[0])
    for i in range(l_nu.shape[0]):
        integrand = l_nu[i, mask][order] / (H_PLANCK * nu_s)
        q_h[i] = np.trapezoid(integrand, nu_s)
    return ages_yr, q_h


fig, ax = plt.subplots(figsize=(6.6, 4.4))

# `except (FileNotFoundError, Exception)` used to guard this: a tuple whose
# second member subsumes the first, so it read as "handle a missing grid" while
# catching everything. Missing grids are the expected case here -- this example
# is skipped in CI precisely because the libraries it compares are not shipped
# -- but a corrupt file or a loader bug was being skipped just as quietly.
plotted = 0
first_failure: Exception | None = None

for (ssp_name, label), color in zip(GRIDS, COLORS):
    try:
        ssp = tengri.load_ssp(ssp_name)
    except Exception as e:
        if first_failure is None:
            first_failure = e
        continue
    ages, q_h = _q_h_per_age(ssp)
    ok = np.isfinite(q_h) & (q_h > 0)
    ax.loglog(ages[ok] / 1.0e6, q_h[ok], color=color, lw=1.6, label=label)
    plotted += 1

if plotted == 0:
    raise RuntimeError(
        f"none of the {len(GRIDS)} SSP grids loaded, so this comparison is "
        f"empty — see the example header for which grids it needs. First "
        f"failure: {type(first_failure).__name__}: {first_failure}"
    ) from first_failure

ax.set_xlim(0.5, 5e3)
ax.set_ylim(1e42, 1e48)
ax.set_xlabel(r"SSP age  [Myr]")
ax.set_ylabel(r"$Q_H$ per unit stellar mass  [s$^{-1}\,M_\odot^{-1}$]")
ax.axvspan(0.5, 10.0, color="0.93", alpha=0.6, lw=0)
ax.text(0.6, 1.5e42, "O-star epoch", color="0.4", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper right")

fig.tight_layout()
plt.savefig("plot_ionizing_lum.png", dpi=150, bbox_inches="tight")